# Baseline: SASRec

SASRec (Kang & McAuley, 2018) con el port en PyTorch [pmixer/SASRec.pytorch](https://github.com/pmixer/SASRec.pytorch) (commit `9f6a02b`), entrenado con el protocolo de LatentR3: Adam, grilla de lr × weight decay, BCE con negativos aleatorios (Apéndice B) y contexto de hasta 10 ítems (Apéndice A). Cada fila de `train` es una muestra (historial de hasta 10 ítems → siguiente ítem).

Se corren cuatro variantes (BCE (paper), CE catálogo, BCE todas las posiciones y CE todas las posiciones, una celda de grilla cada una) y se reportan HR y NDCG @5 y @10 por id y por título; la comparable con la Tabla 1 es BCE (paper) por título.

El entrenamiento está en `sasrec_baseline.py` y la carga de datos, las semillas y las métricas en `../baseline_comun.py`. En local lee los datos de `../../repo_latentr3/data` y el port de `../../SASRec.pytorch`; en Colab, de `data_amazon_2018/` y `SASRec.pytorch/` en el directorio de trabajo, donde también deben estar los dos `.py`. No escribe archivos.

Decisiones, costos y resultados: [README.md](README.md).

## Importaciones

In [1]:
import os
import time
import itertools
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import torch

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


In [2]:
N_PROCESOS = 6   # Se usó 6 sin la medición de paralelismo (D17, pendiente): ../README.md#decisiones-comunes-d1d17
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

cuda AMD Radeon RX 9070 XT


## Datos y código del port

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [4]:
import sys
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3', 'data')
codigo = os.path.abspath('SASRec.pytorch' if IN_COLAB else os.path.join('..', '..', 'SASRec.pytorch'))
# baseline_comun.py está en notebooks/ en local y en el directorio de trabajo en Colab. Los procesos de
# la grilla heredan sys.path, así que también lo encuentran
comun = os.path.abspath('.' if IN_COLAB else '..')
if comun not in sys.path:
    sys.path.insert(0, comun)
names = {'CDs_and_Vinyl': 'cd_data',
         'Musical_Instruments': 'music_data',
         'Toys_and_Games': 'toys_data',
         'Video_Games': 'games_data'}

if not os.path.isfile(os.path.join(codigo, 'python', 'model.py')):
    raise FileNotFoundError(f'No se encontró python/model.py en {codigo}. Descargar '
                            'https://github.com/pmixer/SASRec.pytorch en esa carpeta y hacer checkout del commit 9f6a02b.')
for modulo in ['sasrec_baseline.py', os.path.join(comun, 'baseline_comun.py')]:
    if not os.path.isfile(modulo):
        raise FileNotFoundError(f'No se encontró {os.path.abspath(modulo)}.')

import sasrec_baseline as sb
sb.inicializar_proceso(codigo)   # verifica el sha256 de model.py antes de importarlo

In [5]:
resumen = {}
for categoria, nombre in names.items():
    datos, titulos, prep = sb.cargar_categoria(base, categoria)
    n_items = len(titulos)
    for split, df in datos.items():
        S, P, y = prep[split]['S'], prep[split]['P'], prep[split]['y'].cpu().numpy()
        largos = df['history_item_id'].map(len).to_numpy()
        no_vacias = S != 0
        # Ninguna secuencia de más de 10 ítems, y cada fila conserva su historial completo y en orden
        assert S.shape == P.shape == (len(df), sb.MAX_LEN) and largos.max() <= sb.MAX_LEN
        assert (no_vacias.sum(1) == largos).all()
        assert all(S[i, sb.MAX_LEN - len(h):].tolist() == [j + 1 for j in h]
                   for i, h in enumerate(df['history_item_id']))
        # P: el ítem siguiente de cada posición; en la última, el objetivo de la fila
        assert (no_vacias == (P != 0)).all() and (P[:, -1] == y + 1).all()
        assert (P[:, :-1][no_vacias[:, :-1]] == S[:, 1:][no_vacias[:, :-1]]).all()
        # 0 es el padding; dentro del modelo los ids van de 1 a n_items
        assert S.min() >= 0 and S.max() <= n_items and y.min() >= 0 and y.max() < n_items
    # Cada fila de train aparece una vez por época
    tamanos = [len(y) for _, _, y in sb.lotes(prep['train'], sb.BATCH_SIZE, np.random.default_rng(0))]
    assert sum(tamanos) == len(datos['train'])
    resumen[nombre] = {**{split: len(df) for split, df in datos.items()}, 'ítems': n_items}
print(resumen)

{'cd_data': {'train': 49251, 'valid': 6156, 'test': 6157, 'ítems': 5841}, 'music_data': {'train': 66500, 'valid': 8312, 'test': 8313, 'ítems': 5030}, 'toys_data': {'train': 53898, 'valid': 6737, 'test': 6738, 'ítems': 6299}, 'games_data': {'train': 75175, 'valid': 9397, 'test': 9397, 'ítems': 5308}}


## Métricas

`hit`, `ndcg`, `dedup_titulos` y `evaluar` están en `../baseline_comun.py`: HR y NDCG @5 y @10 **por id** y **por título** (como `calc.py`), con un solo ítem relevante. `evaluar` verifica que las métricas por título nunca sean menores que por id. El criterio por título se explica en el README de `notebooks/`.

## Modelo SASRec

**No lo especifica el paper:** dimensión 50, 2 bloques, 1 cabeza, dropout 0.5, batch de 128 y Adam con betas (0.9, 0.98), que son los valores por defecto del SASRec original en TensorFlow. Además: un negativo uniforme por fila, hasta 200 épocas con early stopping por NDCG@10 (id) en `valid` con paciencia 10, y ranking sobre todo el catálogo de `info/`, igual que los métodos con LLM.

**Del port se usa solo la clase `SASRec`** (`log2feats`, `forward` e `item_emb`), sin modificarla. En `sasrec_baseline.py`:
- `inicializar_proceso`: verifica el sha256 de `python/model.py` y lo importa sin agregar la carpeta del port a `sys.path`.
- `preparar`: rellena los historiales con padding a la izquierda hasta 10, como el sampler original, así que la última posición es siempre el ítem más reciente. El índice 0 queda reservado para el padding, por lo que dentro del modelo los ids van +1.
- `nuevo_modelo`: crea el modelo con la inicialización de `main.py` del port (xavier normal y padding en cero).
- `codificar`: salida de la última posición, como `predict` del port. `puntajes`: producto con los embeddings de todos los ítems del catálogo.
- `perdida_bce`: la del paper, solo en la última posición (el objetivo contra un negativo uniforme).
- `perdida_ce`: referencia, softmax sobre el catálogo.
- `perdida_bce_todas`: referencia, BCE en todas las posiciones del historial con `forward` del port.
- `perdida_ce_todas`: referencia, softmax sobre el catálogo en todas las posiciones del historial, con `log2feats` del port.
- `entrenar`: Adam con weight decay, early stopping en `valid` y vuelta a la mejor época. Alterna los modos train y eval, porque el modelo tiene dropout.
- `correr_combinacion`: entrena una combinación con las semillas fijadas y devuelve su NDCG@10 en `valid`, sus métricas en `test` y el tiempo de entrenamiento.

## Prueba rápida

Entrena 3 épocas de una combinación de CDs. Verifica que el padding siga en cero, que la evaluación sea determinista y que las tres pérdidas den valores finitos. Muestra el tiempo por época y la memoria de GPU, y estima el peor caso de la grilla.

In [6]:
categoria, variante, lr, wd, epocas = 'CDs_and_Vinyl', 'BCE (paper)', 1e-3, 1e-5, 3
datos, titulos, prep = sb.cargar_categoria(base, categoria)
sb.fijar_semillas()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
inicio = time.time()
modelo, epoca, ndcg_valid, corridas_prueba = sb.entrenar(sb.PERDIDAS[variante], lr, wd, prep['train'], prep['valid'],
                                                         datos['valid'], len(titulos), max_epocas=epocas)
t_epoca = (time.time() - inicio) / corridas_prueba

# El padding sigue en cero, el modelo en modo eval es determinista y las tres pérdidas son finitas
assert not modelo.item_emb.weight[0].any() and not modelo.pos_emb.weight[0].any()
recs = sb.recomendar(modelo, prep['test'])
assert recs == sb.recomendar(modelo, prep['test'])
S, P, y = next(sb.lotes(prep['train'], sb.BATCH_SIZE, np.random.default_rng(0)))
with torch.no_grad():
    assert all(torch.isfinite(f(modelo, S, P, y)) for f in sb.PERDIDAS.values())
metricas = sb.evaluar(recs, datos['test'], titulos)   # verifica que por título nunca sea menor que por id

# Peor caso: las 9 combinaciones de cada variante y dataset llegan a 200 épocas. El tiempo por época
# se escala con el número de filas de train; con varios procesos se divide por la aceleración medida
filas = {c: len(sb.cargar_categoria(base, c)[0]['train']) for c in names}
peor = sum(filas[c] / filas[categoria] for c in names) * t_epoca * sb.MAX_EPOCAS * 9 * len(sb.PERDIDAS)
memoria = torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else 0
print(f'{t_epoca:.1f} s por época en CDs (incluye la evaluación en valid), {memoria:.0f} MiB de GPU')
print(f'Peor caso de la grilla con 1 proceso: {peor / 3600:.1f} h')
pd.Series(metricas).round(4)

MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib/python3.12/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1201_32.HIP.fdb.txt"


2.8 s por época en CDs (incluye la evaluación en valid), 98 MiB de GPU
Peor caso de la grilla con 1 proceso: 20.6 h


HR@5 id           0.0158
NDCG@5 id         0.0128
HR@10 id          0.0216
NDCG@10 id        0.0147
HR@5 título       0.0195
NDCG@5 título     0.0150
HR@10 título      0.0286
NDCG@10 título    0.0180
dtype: float64

## Grilla y evaluación

Para cada variante y dataset se entrenan las 9 combinaciones de lr ∈ {1e-2, 1e-3, 1e-4} × weight decay ∈ {1e-4, 1e-5, 1e-6}. Es la misma grilla para todos los baselines tradicionales (Apéndice B). Las combinaciones se reparten en `N_PROCESOS` procesos que comparten la GPU. Cada combinación fija sus semillas antes de empezar, pero la GPU no es exactamente reproducible y no se comprobó que el paralelo dé lo mismo que el secuencial (ver el README).

Las métricas de `test` se calculan en cada proceso, pero la configuración se elige **solo** con el mejor NDCG@10 (id) en `valid`. Ante un empate gana la primera en el orden de la grilla. La variante comparable con la Tabla 1 es **BCE (paper)** con el criterio **por título**.

In [5]:
LRS = [1e-2, 1e-3, 1e-4]
WEIGHT_DECAYS = [1e-4, 1e-5, 1e-6]
corridas = {}   # variante → DataFrame con sus 36 combinaciones, en el orden de la grilla

def correr_grilla(variante):
    """Entrena las 36 combinaciones de una variante (4 datasets × 9) repartidas en N_PROCESOS procesos."""
    combinaciones = list(itertools.product(names, LRS, WEIGHT_DECAYS))
    filas = [None] * len(combinaciones)
    inicio = time.time()
    contexto = multiprocessing.get_context('spawn')   # CUDA no admite procesos creados con fork
    with ProcessPoolExecutor(N_PROCESOS, mp_context=contexto, initializer=sb.inicializar_proceso,
                             initargs=(codigo,)) as pool:
        futuros = {pool.submit(sb.correr_combinacion, base, categoria, variante, lr, wd): i
                   for i, (categoria, lr, wd) in enumerate(combinaciones)}
        for n, futuro in enumerate(as_completed(futuros), 1):
            r = futuro.result()
            filas[futuros[futuro]] = r
            print(f"[{n}/{len(combinaciones)}] {variante} {names[r['categoria']]} lr={r['lr']} wd={r['weight_decay']}: "
                  f"NDCG@10 valid {r['NDCG@10 valid']:.4f} (época {r['época']} de {r['épocas corridas']}) "
                  f"[{(time.time() - inicio) / 60:.1f} min]", flush=True)
    filas = pd.DataFrame(filas)
    filas.insert(1, 'dataset', filas.pop('categoria').map(names))
    return filas

### BCE (paper)

La variante del paper: BCE en la última posición. Es la que se compara con la Tabla 1.

In [8]:
corridas['BCE (paper)'] = correr_grilla('BCE (paper)')

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib/python3.12/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1201_32.HIP.fdb.txt"
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib

[1/36] BCE (paper) cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0071 (época 6 de 16) [1.2 min]
[2/36] BCE (paper) cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0071 (época 6 de 16) [1.3 min]
[3/36] BCE (paper) cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0084 (época 2 de 12) [2.1 min]
[4/36] BCE (paper) cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0085 (época 2 de 12) [2.1 min]
[5/36] BCE (paper) cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0457 (época 24 de 34) [2.5 min]
[6/36] BCE (paper) cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0422 (época 27 de 37) [2.6 min]
[7/36] BCE (paper) cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0084 (época 2 de 12) [2.9 min]
[8/36] BCE (paper) cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0471 (época 30 de 40) [2.9 min]
[9/36] BCE (paper) cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0150 (época 41 de 51) [3.5 min]
[10/36] BCE (paper) music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0491 (época 5 de 15) [3.6 min]
[11/36] BCE (paper) music_data lr=0.01 wd=1e-05: NDCG@10 va

### CE catálogo (referencia)

Softmax sobre todo el catálogo en la última posición. Mide cuánto le cuesta a SASRec la BCE con un negativo.

In [10]:
corridas['CE catálogo'] = correr_grilla('CE catálogo')

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib/python3.12/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1201_32.HIP.fdb.txt"
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib

[1/36] CE catálogo cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0084 (época 5 de 15) [1.1 min]
[2/36] CE catálogo cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0084 (época 9 de 19) [1.5 min]
[3/36] CE catálogo cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0085 (época 10 de 20) [1.5 min]
[4/36] CE catálogo cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0821 (época 20 de 30) [2.1 min]
[5/36] CE catálogo cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0824 (época 20 de 30) [2.2 min]
[6/36] CE catálogo cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0073 (época 1 de 11) [2.2 min]
[7/36] CE catálogo cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0827 (época 20 de 30) [2.2 min]
[8/36] CE catálogo cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0072 (época 1 de 11) [2.3 min]
[9/36] CE catálogo music_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0557 (época 9 de 19) [4.0 min]
[10/36] CE catálogo music_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0493 (época 9 de 19) [4.0 min]
[11/36] CE catálogo music_data lr=0.001 wd=1e-05: NDCG@10 va

### BCE todas las posiciones (referencia)

BCE en cada posición del historial, como el SASRec original. Mide cuánto le cuesta a SASRec entrenar solo en la última posición.

In [9]:
corridas['BCE todas las posiciones'] = correr_grilla('BCE todas las posiciones')

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib/python3.12/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1201_32.HIP.fdb.txt"
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib

[1/36] BCE todas las posiciones cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0094 (época 19 de 29) [2.6 min]
[2/36] BCE todas las posiciones cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0098 (época 19 de 29) [2.6 min]
[3/36] BCE todas las posiciones cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0539 (época 20 de 30) [2.8 min]
[4/36] BCE todas las posiciones cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0326 (época 30 de 40) [3.6 min]
[5/36] BCE todas las posiciones cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0072 (época 2 de 12) [3.6 min]
[6/36] BCE todas las posiciones cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0070 (época 2 de 12) [3.8 min]
[7/36] BCE todas las posiciones cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0522 (época 37 de 47) [3.9 min]
[8/36] BCE todas las posiciones cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0548 (época 37 de 47) [4.1 min]
[9/36] BCE todas las posiciones music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0571 (época 5 de 15) [5.3 min]
[10/36] BCE todas las posiciones mus

### CE todas las posiciones (referencia)

Combina las dos variantes anteriores: softmax sobre todo el catálogo en cada posición del historial, con su contexto causal, y media sobre las posiciones que no son padding. Mide si la supervisión del original suma algo a la CE en la última posición. No se compara con la Tabla 1.

Muestra su propia mejor configuración por dataset (elegida solo con `valid`), así que se puede correr sola después de las celdas de datos y de `correr_grilla`, sin repetir las otras variantes. Las tablas de abajo la incluyen si se corren después de esta celda.

In [6]:
corridas['CE todas las posiciones'] = correr_grilla('CE todas las posiciones')
# Mejor configuración por dataset, elegida solo con valid. lr y weight decay en notación científica,
# porque con round(4) 1e-5 y 1e-6 se verían como 0
c = corridas['CE todas las posiciones']
mejor = c.loc[c.groupby('dataset', sort=False)['NDCG@10 valid'].idxmax()].reset_index(drop=True)
mejor.assign(lr=mejor['lr'].map('{:.0e}'.format),
             weight_decay=mejor['weight_decay'].map('{:.0e}'.format)).round(4)

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib/python3.12/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1201_32.HIP.fdb.txt"
MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib

[1/36] CE todas las posiciones cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0739 (época 14 de 24) [2.0 min]
[2/36] CE todas las posiciones cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0803 (época 14 de 24) [2.0 min]
[3/36] CE todas las posiciones cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0812 (época 21 de 31) [2.5 min]
[4/36] CE todas las posiciones cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0786 (época 30 de 40) [3.2 min]
[5/36] CE todas las posiciones cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0788 (época 35 de 45) [3.6 min]
[6/36] CE todas las posiciones cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0792 (época 44 de 54) [4.4 min]
[7/36] CE todas las posiciones music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.1029 (época 12 de 22) [5.5 min]
[8/36] CE todas las posiciones music_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0682 (época 15 de 25) [6.3 min]
[9/36] CE todas las posiciones music_data lr=0.001 wd=0.0001: NDCG@10 valid 0.1027 (época 3 de 13) [6.8 min]
[10/36] CE todas las posiciones music_d

,variante,dataset,lr,weight_decay,época,épocas corridas,segundos,NDCG@10 valid,HR@5 id,NDCG@5 id,HR@10 id,NDCG@10 id,HR@5 título,NDCG@5 título,HR@10 título,NDCG@10 título
0,CE todas las posiciones,cd_data,1e-02,1e-06,21,31,147.7705,0.0812,0.0997,0.0757,0.1257,0.0841,0.1041,0.0786,0.1338,0.0882
1,CE todas las posiciones,music_data,1e-03,1e-05,4,14,86.2627,0.1037,0.0923,0.0784,0.1150,0.0858,0.0924,0.0786,0.1150,0.0859
2,CE todas las posiciones,toys_data,1e-03,1e-04,24,34,166.3383,0.0744,0.0793,0.0523,0.1064,0.0611,0.0795,0.0525,0.1064,0.0612
3,CE todas las posiciones,games_data,1e-03,1e-06,33,43,293.9250,0.0704,0.0590,0.0376,0.0916,0.0481,0.0590,0.0376,0.0916,0.0481


### Mejor configuración por variante y dataset

Usa las variantes que se hayan corrido, y elige solo con `valid`.

In [7]:
print('Variantes corridas:', list(corridas))
todas = pd.concat(corridas.values(), ignore_index=True)
# Mejor configuración por variante y dataset, elegida solo con valid
resultados = todas.loc[todas.groupby(['variante', 'dataset'], sort=False)['NDCG@10 valid'].idxmax()]
resultados = resultados.reset_index(drop=True)
resultados.round(4)

Variantes corridas: ['CE todas las posiciones']


,variante,dataset,lr,weight_decay,época,épocas corridas,segundos,NDCG@10 valid,HR@5 id,NDCG@5 id,HR@10 id,NDCG@10 id,HR@5 título,NDCG@5 título,HR@10 título,NDCG@10 título
0,CE todas las posiciones,cd_data,0.010,0.0000,21,31,147.7705,0.0812,0.0997,0.0757,0.1257,0.0841,0.1041,0.0786,0.1338,0.0882
1,CE todas las posiciones,music_data,0.001,0.0000,4,14,86.2627,0.1037,0.0923,0.0784,0.1150,0.0858,0.0924,0.0786,0.1150,0.0859
2,CE todas las posiciones,toys_data,0.001,0.0001,24,34,166.3383,0.0744,0.0793,0.0523,0.1064,0.0611,0.0795,0.0525,0.1064,0.0612
3,CE todas las posiciones,games_data,0.001,0.0000,33,43,293.9250,0.0704,0.0590,0.0376,0.0916,0.0481,0.0590,0.0376,0.0916,0.0481


## Comparación con la Tabla 1 del paper

SASRec según el paper, junto a la variante BCE por id y por título y las variantes de referencia por título. Las variantes de referencia no se comparan con la Tabla 1: muestran cuánto cambia SASRec con otra pérdida (CE), con la supervisión del original (todas las posiciones) o con ambas.

In [12]:
PAPER = {'toys_data': [0.0601, 0.0760, 0.0458, 0.0510], 'cd_data': [0.0841, 0.1054, 0.0622, 0.0691],
         'games_data': [0.0416, 0.0633, 0.0280, 0.0350], 'music_data': [0.0793, 0.0950, 0.0708, 0.0758]}
METRICAS = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']
FUENTES = [('BCE (paper), id', 'BCE (paper)', 'id'), ('BCE (paper), título', 'BCE (paper)', 'título'),
           ('CE catálogo, título', 'CE catálogo', 'título'),
           ('BCE todas las posiciones, título', 'BCE todas las posiciones', 'título'),
           ('CE todas las posiciones, título', 'CE todas las posiciones', 'título')]

def fila(variante, nombre, criterio):
    r = resultados[(resultados['variante'] == variante) & (resultados['dataset'] == nombre)].iloc[0]
    return [r[f'{m} {criterio}'] for m in METRICAS]

tabla = []
for nombre in names.values():
    tabla.append({'dataset': nombre, 'fuente': 'Paper', **dict(zip(METRICAS, PAPER[nombre]))})
    for fuente, variante, criterio in FUENTES:
        if variante in corridas:   # solo las variantes que se corrieron
            tabla.append({'dataset': nombre, 'fuente': fuente,
                          **dict(zip(METRICAS, fila(variante, nombre, criterio)))})
pd.DataFrame(tabla).round(4)

,dataset,fuente,HR@5,HR@10,NDCG@5,NDCG@10
0,cd_data,Paper,0.0841,0.1054,0.0622,0.0691
1,cd_data,"BCE (paper), id",0.0640,0.0903,0.0409,0.0493
2,cd_data,"BCE (paper), título",0.0697,0.0986,0.0466,0.0559
3,cd_data,"CE catálogo, título",0.1145,0.1493,0.0812,0.0925
4,cd_data,"BCE todas las posiciones, título",0.0872,0.1192,0.0551,0.0653
5,music_data,Paper,0.0793,0.0950,0.0708,0.0758
6,music_data,"BCE (paper), id",0.0764,0.1030,0.0423,0.0509
7,music_data,"BCE (paper), título",0.0765,0.1033,0.0425,0.0512
8,music_data,"CE catálogo, título",0.0994,0.1209,0.0840,0.0909
9,music_data,"BCE todas las posiciones, título",0.0826,0.1022,0.0465,0.0527


## Pisos de referencia

MostPop por frecuencia en train (HR@10 y NDCG@10 en test; ver la sección de pisos de [../README.md](../README.md#pisos-de-referencia-mostpop)). Si la variante BCE (paper) queda por debajo, probablemente hay un error.

In [ ]:
PISOS = {'cd_data': (0.0197, 0.0091), 'music_data': (0.0659, 0.0330),
         'toys_data': (0.0138, 0.0082), 'games_data': (0.0208, 0.0111)}
assert 'BCE (paper)' in corridas, 'Falta correr la celda de la variante BCE (paper)'
for nombre, (piso_hr, piso_ndcg) in PISOS.items():
    r = resultados[(resultados['variante'] == 'BCE (paper)') & (resultados['dataset'] == nombre)].iloc[0]
    ok = r['HR@10 id'] >= piso_hr and r['NDCG@10 id'] >= piso_ndcg
    print(f"{nombre}: HR@10 id {r['HR@10 id']:.4f} (piso {piso_hr}), "
          f"NDCG@10 id {r['NDCG@10 id']:.4f} (piso {piso_ndcg}): {'OK' if ok else 'ALERTA'}")

cd_data: HR@10 id 0.0903 (piso 0.0197), NDCG@10 id 0.0493 (piso 0.0091): OK
music_data: HR@10 id 0.1030 (piso 0.0659), NDCG@10 id 0.0509 (piso 0.033): OK
toys_data: HR@10 id 0.0772 (piso 0.0138), NDCG@10 id 0.0432 (piso 0.0082): OK
games_data: HR@10 id 0.0687 (piso 0.0208), NDCG@10 id 0.0350 (piso 0.0111): OK


: 